In [2]:
# ============================================================
# Cell 1 - Library dan konfigurasi utama
# ============================================================

from pathlib import Path
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings("ignore")


# ============================================================
# ROOT PATH
# ============================================================

SAMPLED_ROOT = Path("/media/spell/Spell-lab/Lidar/E.Sampled Dataset")
NPZ_ROOT = Path("/media/spell/Spell-lab/Lidar/G.NPZ Dataset")


# ============================================================
# AUDIT CONFIG
# ============================================================

NPZ_COLUMNS = [
    "Timestamp",
    "X",
    "Y",
    "Z",
    "X_corr",
    "Y_corr",
    "Z_corr",
    "Z_level",
    "Reflectivity",
]

FRAME_ID_COLUMN = "frame_id"
REQUIRED_CSV_COLUMNS = [FRAME_ID_COLUMN] + NPZ_COLUMNS

FLOAT_TOLERANCE = 1e-5

# Jika None, audit semua file NPZ yang ditemukan.
# Jika ingin sample saja, isi angka, misalnya 50.
N_AUDIT_SAMPLE = None

RANDOM_STATE = 42

AUDIT_OUT_DIR = NPZ_ROOT / "_audit_csv_vs_npz"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("===== CSV VS NPZ AUDIT CONFIG =====")
print(f"Sampled CSV root : {SAMPLED_ROOT}")
print(f"NPZ root         : {NPZ_ROOT}")
print(f"Audit output dir : {AUDIT_OUT_DIR}")
print(f"Audit sample     : {N_AUDIT_SAMPLE}")
print(f"Tolerance        : {FLOAT_TOLERANCE}")

===== CSV VS NPZ AUDIT CONFIG =====
Sampled CSV root : /media/spell/Spell-lab/Lidar/E.Sampled Dataset
NPZ root         : /media/spell/Spell-lab/Lidar/G.NPZ Dataset
Audit output dir : /media/spell/Spell-lab/Lidar/G.NPZ Dataset/_audit_csv_vs_npz
Audit sample     : None
Tolerance        : 1e-05


In [3]:
# ============================================================
# Cell 2 - Build audit manifest dari NPZ
# ============================================================

npz_paths = sorted([
    p for p in NPZ_ROOT.rglob("*.npz")
    if "_audit_csv_vs_npz" not in str(p)
])

if len(npz_paths) == 0:
    raise FileNotFoundError(f"Tidak ada file NPZ ditemukan di: {NPZ_ROOT}")

records = []

for npz_path in npz_paths:
    try:
        loaded = np.load(npz_path, allow_pickle=True)
        source_csv = Path(loaded["source_csv"].item())
        n_folder = loaded["n_folder"].item() if "n_folder" in loaded.files else npz_path.parts[-5]
    except Exception as e:
        source_csv = None
        n_folder = None

    records.append({
        "npz_path": npz_path,
        "source_csv": source_csv,
        "n_folder": n_folder,
        "npz_exists": npz_path.exists(),
        "csv_exists": source_csv.exists() if source_csv is not None else False,
    })

audit_manifest_df = pd.DataFrame(records)

if N_AUDIT_SAMPLE is not None and N_AUDIT_SAMPLE < len(audit_manifest_df):
    audit_manifest_df = audit_manifest_df.sample(
        N_AUDIT_SAMPLE,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

manifest_save = audit_manifest_df.copy()
manifest_save["npz_path"] = manifest_save["npz_path"].astype(str)
manifest_save["source_csv"] = manifest_save["source_csv"].astype(str)

manifest_path = AUDIT_OUT_DIR / "audit_manifest.csv"
manifest_save.to_csv(manifest_path, index=False)

print("===== AUDIT MANIFEST =====")
print(f"Total NPZ files found : {len(npz_paths)}")
print(f"Files to audit        : {len(audit_manifest_df)}")
print(f"Manifest saved        : {manifest_path}")

display(audit_manifest_df.head())

===== AUDIT MANIFEST =====
Total NPZ files found : 3960
Files to audit        : 3960
Manifest saved        : /media/spell/Spell-lab/Lidar/G.NPZ Dataset/_audit_csv_vs_npz/audit_manifest.csv


,npz_path,source_csv,n_folder,npz_exists,csv_exists
0,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,N0256,True,True
1,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,N0256,True,True
2,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,N0256,True,True
3,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,N0256,True,True
4,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,N0256,True,True


In [4]:
# ============================================================
# Cell 3 - Helper reconstruct CSV array
# ============================================================

def load_csv_as_frame_array(csv_path):
    df = pd.read_csv(csv_path)

    missing_cols = [col for col in REQUIRED_CSV_COLUMNS if col not in df.columns]
    if missing_cols:
        raise ValueError(f"CSV missing columns: {missing_cols}")

    df = df[REQUIRED_CSV_COLUMNS].copy()

    for col in REQUIRED_CSV_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=REQUIRED_CSV_COLUMNS).copy()
    df[FRAME_ID_COLUMN] = df[FRAME_ID_COLUMN].astype(int)

    df = df.sort_values([FRAME_ID_COLUMN]).reset_index(drop=True)

    frame_counts = df.groupby(FRAME_ID_COLUMN).size()
    unique_counts = sorted(frame_counts.unique().tolist())

    if len(unique_counts) != 1:
        raise ValueError(f"CSV variable points per frame: {unique_counts[:10]}")

    n_points = int(unique_counts[0])
    frame_ids = frame_counts.index.to_numpy(dtype=np.int32)

    frames = []

    for fid in frame_ids:
        frame_df = df[df[FRAME_ID_COLUMN] == fid]
        arr = frame_df[NPZ_COLUMNS].to_numpy(dtype=np.float32)
        frames.append(arr)

    data = np.stack(frames, axis=0).astype(np.float32)

    return data, frame_ids, df

In [5]:
# ============================================================
# Cell 4 - Helper audit satu file
# ============================================================

def audit_one_csv_npz(npz_path):
    npz_path = Path(npz_path)

    result = {
        "npz_path": str(npz_path),
        "source_csv": "",
        "status": "unknown",
        "shape_match": False,
        "frame_ids_match": False,
        "feature_names_match": False,
        "value_match": False,
        "metadata_match": False,
        "max_abs_diff": np.nan,
        "mean_abs_diff": np.nan,
        "csv_shape": "",
        "npz_shape": "",
        "n_frames": 0,
        "n_points": 0,
    }

    try:
        loaded = np.load(npz_path, allow_pickle=True)

        source_csv = Path(loaded["source_csv"].item())
        result["source_csv"] = str(source_csv)

        if not source_csv.exists():
            result["status"] = "source_csv_missing"
            return result

        npz_data = loaded["data"].astype(np.float32)
        npz_frame_ids = loaded["frame_ids"].astype(np.int32)
        npz_feature_names = list(loaded["feature_names"])

        csv_data, csv_frame_ids, csv_df = load_csv_as_frame_array(source_csv)

        result["csv_shape"] = str(csv_data.shape)
        result["npz_shape"] = str(npz_data.shape)
        result["n_frames"] = int(npz_data.shape[0]) if npz_data.ndim == 3 else 0
        result["n_points"] = int(npz_data.shape[1]) if npz_data.ndim == 3 else 0

        shape_match = csv_data.shape == npz_data.shape
        frame_ids_match = np.array_equal(csv_frame_ids, npz_frame_ids)
        feature_names_match = npz_feature_names == NPZ_COLUMNS

        result["shape_match"] = bool(shape_match)
        result["frame_ids_match"] = bool(frame_ids_match)
        result["feature_names_match"] = bool(feature_names_match)

        if shape_match:
            abs_diff = np.abs(csv_data - npz_data)
            max_abs_diff = float(np.max(abs_diff))
            mean_abs_diff = float(np.mean(abs_diff))

            result["max_abs_diff"] = max_abs_diff
            result["mean_abs_diff"] = mean_abs_diff
            result["value_match"] = bool(max_abs_diff <= FLOAT_TOLERANCE)
        else:
            result["value_match"] = False

        # Metadata basic check dari path source csv dan NPZ.
        # Karena source_csv adalah path asli, metadata dianggap match jika file path valid
        # dan field metadata utama tersedia.
        required_meta_keys = ["dataset", "room", "activity", "subject", "file_id", "source_csv"]

        metadata_keys_available = all(k in loaded.files for k in required_meta_keys)

        if metadata_keys_available:
            source_csv_from_npz = Path(loaded["source_csv"].item())
            result["metadata_match"] = bool(source_csv_from_npz == source_csv)
        else:
            result["metadata_match"] = False

        all_pass = (
            result["shape_match"]
            and result["frame_ids_match"]
            and result["feature_names_match"]
            and result["value_match"]
            and result["metadata_match"]
        )

        result["status"] = "PASS" if all_pass else "FAIL"

        return result

    except Exception as e:
        result["status"] = f"ERROR: {e}"
        return result

In [6]:
# ============================================================
# Cell 5 - Jalankan audit semua file
# ============================================================

audit_results = []

for _, row in tqdm(audit_manifest_df.iterrows(), total=len(audit_manifest_df), desc="Auditing CSV vs NPZ"):
    result = audit_one_csv_npz(row["npz_path"])
    audit_results.append(result)

audit_results_df = pd.DataFrame(audit_results)

audit_results_path = AUDIT_OUT_DIR / "csv_vs_npz_audit_results.csv"
audit_results_df.to_csv(audit_results_path, index=False)

print("===== AUDIT DONE =====")
print(f"Audit results saved: {audit_results_path}")

display(audit_results_df["status"].value_counts(dropna=False).to_frame("count"))
display(audit_results_df.head())

Auditing CSV vs NPZ:   0%|          | 0/3960 [00:00<?, ?it/s]

===== AUDIT DONE =====
Audit results saved: /media/spell/Spell-lab/Lidar/G.NPZ Dataset/_audit_csv_vs_npz/csv_vs_npz_audit_results.csv


,count
status,
PASS,3960


,npz_path,source_csv,status,shape_match,frame_ids_match,feature_names_match,value_match,metadata_match,max_abs_diff,mean_abs_diff,csv_shape,npz_shape,n_frames,n_points
0,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,PASS,True,True,True,True,True,0.0,0.0,"(55, 256, 9)","(55, 256, 9)",55,256
1,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,PASS,True,True,True,True,True,0.0,0.0,"(54, 256, 9)","(54, 256, 9)",54,256
2,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,PASS,True,True,True,True,True,0.0,0.0,"(56, 256, 9)","(56, 256, 9)",56,256
3,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,PASS,True,True,True,True,True,0.0,0.0,"(58, 256, 9)","(58, 256, 9)",58,256
4,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...,/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,PASS,True,True,True,True,True,0.0,0.0,"(56, 256, 9)","(56, 256, 9)",56,256


In [7]:
# ============================================================
# Cell 6 - Ringkasan audit global
# ============================================================

total_files = len(audit_results_df)
pass_files = int((audit_results_df["status"] == "PASS").sum())
fail_files = total_files - pass_files

summary = {
    "audit_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_files_audited": total_files,
    "pass_files": pass_files,
    "fail_or_error_files": fail_files,
    "pass_ratio": pass_files / total_files if total_files > 0 else np.nan,
    "float_tolerance": FLOAT_TOLERANCE,
    "max_of_max_abs_diff": audit_results_df["max_abs_diff"].max(),
    "mean_of_mean_abs_diff": audit_results_df["mean_abs_diff"].mean(),
}

audit_summary_df = pd.DataFrame([summary])

audit_summary_path = AUDIT_OUT_DIR / "csv_vs_npz_audit_summary.csv"
audit_summary_json_path = AUDIT_OUT_DIR / "csv_vs_npz_audit_summary.json"

audit_summary_df.to_csv(audit_summary_path, index=False)

with open(audit_summary_json_path, "w") as f:
    json.dump(summary, f, indent=4)

print("===== GLOBAL AUDIT SUMMARY =====")
display(audit_summary_df)

print(f"Summary CSV : {audit_summary_path}")
print(f"Summary JSON: {audit_summary_json_path}")

===== GLOBAL AUDIT SUMMARY =====


,audit_time,total_files_audited,pass_files,fail_or_error_files,pass_ratio,float_tolerance,max_of_max_abs_diff,mean_of_mean_abs_diff
0,2026-05-11 20:16:32,3960,3960,0,1.0,0.00001,0.0,0.0


Summary CSV : /media/spell/Spell-lab/Lidar/G.NPZ Dataset/_audit_csv_vs_npz/csv_vs_npz_audit_summary.csv
Summary JSON: /media/spell/Spell-lab/Lidar/G.NPZ Dataset/_audit_csv_vs_npz/csv_vs_npz_audit_summary.json


In [8]:
# ============================================================
# Cell 7 - Tampilkan file gagal jika ada
# ============================================================

failed_df = audit_results_df[audit_results_df["status"] != "PASS"].copy()

print(f"Failed/Error files: {len(failed_df)}")

if len(failed_df) > 0:
    failed_path = AUDIT_OUT_DIR / "csv_vs_npz_failed_files.csv"
    failed_df.to_csv(failed_path, index=False)

    display(failed_df.head(50))
    print(f"Failed files saved to: {failed_path}")
else:
    print("All audited files PASSED.")

Failed/Error files: 0
All audited files PASSED.


In [9]:
# ============================================================
# Cell 8 - Cek distribusi max_abs_diff
# ============================================================

valid_diff_df = audit_results_df[
    audit_results_df["status"].isin(["PASS", "FAIL"])
].copy()

if len(valid_diff_df) > 0:
    print("===== MAX ABS DIFF SUMMARY =====")
    display(valid_diff_df["max_abs_diff"].describe().to_frame("max_abs_diff"))

    print("===== WORST MAX ABS DIFF FILES =====")
    display(
        valid_diff_df
        .sort_values("max_abs_diff", ascending=False)
        .head(20)
        [
            [
                "status",
                "max_abs_diff",
                "mean_abs_diff",
                "csv_shape",
                "npz_shape",
                "source_csv",
                "npz_path",
            ]
        ]
    )

===== MAX ABS DIFF SUMMARY =====


,max_abs_diff
count,3960.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


===== WORST MAX ABS DIFF FILES =====


,status,max_abs_diff,mean_abs_diff,csv_shape,npz_shape,source_csv,npz_path
0,PASS,0.0,0.0,"(55, 256, 9)","(55, 256, 9)",/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N02...
2630,PASS,0.0,0.0,"(57, 512, 9)","(57, 512, 9)",/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N05...
2632,PASS,0.0,0.0,"(65, 512, 9)","(65, 512, 9)",/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N05...
2633,PASS,0.0,0.0,"(67, 512, 9)","(67, 512, 9)",/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N05...
2634,PASS,0.0,0.0,"(82, 512, 9)","(82, 512, 9)",/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N05...
2635,PASS,0.0,0.0,"(59, 512, 9)","(59, 512, 9)",/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N05...
2636,PASS,0.0,0.0,"(61, 512, 9)","(61, 512, 9)",/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N05...
2637,PASS,0.0,0.0,"(59, 512, 9)","(59, 512, 9)",/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N05...
2638,PASS,0.0,0.0,"(58, 512, 9)","(58, 512, 9)",/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N05...
2639,PASS,0.0,0.0,"(55, 512, 9)","(55, 512, 9)",/media/spell/Spell-lab/Lidar/E.Sampled Dataset...,/media/spell/Spell-lab/Lidar/G.NPZ Dataset/N05...


In [10]:
# ============================================================
# Cell 9 - Final strict assertion
# ============================================================

assert fail_files == 0, f"Audit failed: {fail_files} file(s) failed or errored. Check failed report."

print("CSV vs NPZ audit PASSED for all audited files.")

CSV vs NPZ audit PASSED for all audited files.
